# DUNS ID Enrichment Notebook

**Purpose:** Load an Excel file from `data/input/excel_files/`, match it against the unified DUNS reference (`data/output/unified_duns_list.csv`) using **client number ↔ CS**, append DUNS IDs, and produce a full summary report.

| Section | What |
|---|---|
| 1 | Imports & paths |
| 2 | Load input Excel file |
| 3 | Load unified DUNS reference |
| 4 | Configure join keys |
| 5 | Match & enrich |
| 6 | **Summary — match rate, unmatched samples** |
| 7 | Export enriched file & review package |

## Section 1 — Imports & Paths

In [ ]:
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd

warnings.filterwarnings("ignore")

# ── Root paths ────────────────────────────────────────────────────────────────
NOTEBOOK_DIR     = Path().resolve()                           # duns_id/notebooks/
PROJECT_ROOT     = NOTEBOOK_DIR.parent                        # duns_id/

INPUT_EXCEL_DIR  = PROJECT_ROOT / "data" / "input" / "excel_files"
REFERENCE_DIR    = PROJECT_ROOT / "data" / "output"
OUTPUT_DIR       = PROJECT_ROOT / "data" / "output"
REVIEW_DIR       = PROJECT_ROOT / "review"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

# ── ✏️  UPDATE THESE TWO LINES ────────────────────────────────────────────────
EXCEL_FILENAME   = "your_input_file.xlsx"   # filename inside data/input/excel_files/
EXCEL_SHEET      = 0                        # sheet index (0 = first) or sheet name
# ─────────────────────────────────────────────────────────────────────────────

DUNS_REFERENCE_FILE = REFERENCE_DIR / "unified_duns_list.csv"
EXCEL_FILE          = INPUT_EXCEL_DIR / EXCEL_FILENAME
RUN_TS              = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

print(f"Input Excel   : {EXCEL_FILE}")
print(f"DUNS reference: {DUNS_REFERENCE_FILE}")
print(f"Output dir    : {OUTPUT_DIR}")

## Section 2 — Load Input Excel File

In [ ]:
if not EXCEL_FILE.exists():
    raise FileNotFoundError(
        f"\nExcel file not found: {EXCEL_FILE}"
        f"\n → Place your file in: {INPUT_EXCEL_DIR}"
        f"\n → Then update EXCEL_FILENAME in Section 1."
    )

excel_df = pd.read_excel(EXCEL_FILE, sheet_name=EXCEL_SHEET, dtype=str)

# Normalise: strip whitespace from all string columns
excel_df = excel_df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print(f"Rows loaded : {len(excel_df):,}")
print(f"Columns ({len(excel_df.columns)}): {list(excel_df.columns)}")
excel_df.head(5)

## Section 3 — Load Unified DUNS Reference

In [ ]:
if not DUNS_REFERENCE_FILE.exists():
    raise FileNotFoundError(
        f"\nDUNS reference not found: {DUNS_REFERENCE_FILE}"
        "\n → Run scripts/consolidate_duns.py first to generate it."
    )

duns_ref = pd.read_csv(DUNS_REFERENCE_FILE, dtype=str)
duns_ref = duns_ref.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print(f"DUNS reference rows : {len(duns_ref):,}")
print(f"Columns ({len(duns_ref.columns)})  : {list(duns_ref.columns)}")
duns_ref.head(5)

## Section 4 — Configure Join Keys

Set `EXCEL_CLIENT_COL` to the exact column name in your Excel file that holds the **client / customer number**.  
This will be matched against the `CS` column in the DUNS reference.

In [ ]:
# ── ✏️  UPDATE THIS ───────────────────────────────────────────────────────────
EXCEL_CLIENT_COL = "client_number"   # column in your Excel file to match on CS
# ─────────────────────────────────────────────────────────────────────────────

# Validate the column exists
if EXCEL_CLIENT_COL not in excel_df.columns:
    raise KeyError(
        f"\nColumn '{EXCEL_CLIENT_COL}' not found in Excel file."
        f"\nAvailable columns: {list(excel_df.columns)}"
        f"\nUpdate EXCEL_CLIENT_COL above."
    )

if "CS" not in duns_ref.columns:
    raise KeyError("'CS' column not found in DUNS reference. Check unified_duns_list.csv.")

print(f"Joining  Excel['{EXCEL_CLIENT_COL}']  →  DUNS ref['CS']")
print(f"Excel unique clients : {excel_df[EXCEL_CLIENT_COL].nunique():,}")
print(f"DUNS ref unique CS   : {duns_ref['CS'].nunique():,}")

## Section 5 — Match & Enrich

In [ ]:
# Columns to bring in from the DUNS reference
DUNS_COLS = ["CS", "duns", "duns_count", "BU", "IDS", "SDS"]
duns_ref_slim = duns_ref[[c for c in DUNS_COLS if c in duns_ref.columns]].copy()

# Left join: keep all Excel rows, append DUNS where found
enriched = excel_df.merge(
    duns_ref_slim.rename(columns={"CS": EXCEL_CLIENT_COL}),
    on=EXCEL_CLIENT_COL,
    how="left",
    suffixes=("", "_duns")
)

# Flag each row
enriched["duns_match_status"] = enriched["duns"].apply(
    lambda x: "MATCHED" if pd.notna(x) and str(x).strip() not in ("", "nan") else "UNMATCHED"
)

print(f"Total rows after join : {len(enriched):,}")
print(f"MATCHED               : {(enriched['duns_match_status'] == 'MATCHED').sum():,}")
print(f"UNMATCHED             : {(enriched['duns_match_status'] == 'UNMATCHED').sum():,}")
enriched.head(5)

## Section 6 — Match Summary

In [ ]:
total      = len(enriched)
matched    = (enriched["duns_match_status"] == "MATCHED").sum()
unmatched  = total - matched
pct_match  = matched / total * 100 if total else 0
pct_unmatch = 100 - pct_match

# ── High-level table ────────────────────────────────────────────────────────
summary_df = pd.DataFrame({
    "Category"  : ["Total clients", "Matched (DUNS found)", "Unmatched (no DUNS)"],
    "Count"     : [total, matched, unmatched],
    "Percentage": [100.0, round(pct_match, 2), round(pct_unmatch, 2)],
})
print("=" * 45)
print("         MATCH SUMMARY REPORT")
print("=" * 45)
print(summary_df.to_string(index=False))
print("=" * 45)

# ── Clients with multiple DUNS IDs ──────────────────────────────────────────
if "duns_count" in enriched.columns:
    multi_duns = enriched[
        enriched["duns_match_status"] == "MATCHED"
    ]["duns_count"].apply(lambda x: int(x) if pd.notna(x) else 0)
    multi_flag = (multi_duns > 1).sum()
    print(f"\nClients with multiple DUNS IDs : {multi_flag:,}")

# ── Sample of unmatched rows ─────────────────────────────────────────────────
unmatched_df = enriched[enriched["duns_match_status"] == "UNMATCHED"].reset_index(drop=True)

print(f"\nSample of unmatched clients (up to 10):")
display(unmatched_df[[EXCEL_CLIENT_COL]].drop_duplicates().head(10))

## Section 7 — Export Results